https://www.kaggle.com/competitions/aicc-round-0-brain-tumor

In [ ]:
import os
from itertools import cycle
import pandas as pd
from PIL import Image
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms as T
from torchvision import models

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
NUM_CLASSES = 4
EPOCHS = 25
LABELED_BATCH = 8
UNLABELED_BATCH = 56
LR = 3e-4
LAMBDA_U = 1.0
INIT_THRESHOLD = 0.70
FINAL_THRESHOLD = 0.95

TRAIN_CSV = "/kaggle/input/competitions/aicc-round-0-brain-tumor/train.csv"
TRAIN_DIR = "/kaggle/input/competitions/aicc-round-0-brain-tumor/train"
TEST_DIR = "/kaggle/input/competitions/aicc-round-0-brain-tumor/test"

weak_transform = T.Compose([
    T.Resize((224, 224)),
    T.RandomHorizontalFlip(),
    T.ToTensor(),
    T.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

strong_transform = T.Compose([
    T.Resize((224, 224)),
    T.RandomHorizontalFlip(),
    T.RandomRotation(20),
    T.ColorJitter(brightness=0.4, contrast=0.4, saturation=0.4),
    T.RandomAffine(degrees=0, translate=(0.1, 0.1), scale=(0.8, 1.2), shear=10),
    T.RandomGrayscale(p=0.1),
    T.ToTensor(),
    T.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

class LabeledDataset(Dataset):
    def __init__(self, csv_path, image_dir, transform):
        self.df = pd.read_csv(csv_path)
        self.image_dir = image_dir
        self.transform = transform
        unique_labels = sorted(self.df["label"].unique())
        self.label_to_idx = {name: i for i, name in enumerate(unique_labels)}
        self.idx_to_label = {i: name for i, name in enumerate(unique_labels)}

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        image_id = row["image_id"]
        label_str = row["label"]
        label = self.label_to_idx[label_str]
        filename = f"{image_id:04d}.jpg"
        path = os.path.join(self.image_dir, filename)
        image = Image.open(path).convert("RGB")
        image = self.transform(image)
        return image, label

class UnlabeledDataset(Dataset):
    def __init__(self, image_dir, labeled_ids, weak_transform, strong_transform):
        self.image_dir = image_dir
        self.weak_transform = weak_transform
        self.strong_transform = strong_transform
        self.files = []
        for f in sorted(os.listdir(image_dir)):
            img_id = int(f.split(".")[0])
            if img_id not in labeled_ids:
                self.files.append(f)

    def __len__(self):
        return len(self.files)

    def __getitem__(self, idx):
        filename = self.files[idx]
        path = os.path.join(self.image_dir, filename)
        image = Image.open(path).convert("RGB")
        weak_img = self.weak_transform(image)
        strong_img = self.strong_transform(image)
        return weak_img, strong_img

class TestDataset(Dataset):
    def __init__(self, image_dir, transform):
        self.image_dir = image_dir
        self.transform = transform
        self.files = sorted(os.listdir(image_dir))

    def __len__(self):
        return len(self.files)

    def __getitem__(self, idx):
        filename = self.files[idx]
        path = os.path.join(self.image_dir, filename)
        image = Image.open(path).convert("RGB")
        image = self.transform(image)
        image_id = int(filename.split(".")[0])
        return image, image_id

train_df = pd.read_csv(TRAIN_CSV)
labeled_ids = set(train_df["image_id"])

labeled_dataset = LabeledDataset(TRAIN_CSV, TRAIN_DIR, weak_transform)
unlabeled_dataset = UnlabeledDataset(TRAIN_DIR, labeled_ids, weak_transform, strong_transform)
test_dataset = TestDataset(TEST_DIR, weak_transform)

labeled_loader = DataLoader(labeled_dataset, batch_size=LABELED_BATCH, shuffle=True, num_workers=2)
unlabeled_loader = DataLoader(unlabeled_dataset, batch_size=UNLABELED_BATCH, shuffle=True, num_workers=2)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False, num_workers=2)

print("Labeled:", len(labeled_dataset))
print("Unlabeled:", len(unlabeled_dataset))

model = models.resnet18(weights='DEFAULT')
model.fc = nn.Linear(model.fc.in_features, NUM_CLASSES)
model = model.to(DEVICE)

optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
best_loss = 999999

for epoch in range(EPOCHS):
    model.train()
    running_loss = 0
    running_supervised = 0
    running_unsupervised = 0
    running_mask = 0

    threshold = INIT_THRESHOLD + (FINAL_THRESHOLD - INIT_THRESHOLD) * (epoch / EPOCHS)
    labeled_iter = cycle(labeled_loader)

    for weak_imgs, strong_imgs in unlabeled_loader:
        images_l, labels_l = next(labeled_iter)
        
        if isinstance(labels_l, tuple) or isinstance(labels_l, list):
            labels_l = torch.tensor(labels_l)

        images_l = images_l.to(DEVICE)
        labels_l = labels_l.to(DEVICE).long()
        weak_imgs = weak_imgs.to(DEVICE)
        strong_imgs = strong_imgs.to(DEVICE)

        preds_l = model(images_l)
        loss_l = F.cross_entropy(preds_l, labels_l)

        with torch.no_grad():
            weak_preds = model(weak_imgs)
            probs = F.softmax(weak_preds, dim=1)
            confidence, pseudo_labels = torch.max(probs, dim=1)
            mask = (confidence > threshold).float()

        strong_preds = model(strong_imgs)
        loss_u = (F.cross_entropy(strong_preds, pseudo_labels, reduction='none') * mask).mean()

        total_loss = loss_l + LAMBDA_U * loss_u

        optimizer.zero_grad()
        total_loss.backward()
        optimizer.step()

        running_loss += total_loss.item()
        running_supervised += loss_l.item()
        running_unsupervised += loss_u.item()
        running_mask += mask.mean().item()

    avg_loss = running_loss / len(unlabeled_loader)
    avg_mask = running_mask / len(unlabeled_loader)

    print(
        f"Epoch [{epoch+1}/{EPOCHS}] "
        f"| Total:{avg_loss:.4f} "
        f"| Sup:{running_supervised/len(unlabeled_loader):.4f} "
        f"| Unsup:{running_unsupervised/len(unlabeled_loader):.4f} "
        f"| Threshold:{threshold:.3f} "
        f"| PseudoUsed:{avg_mask:.3f}"
    )

    if avg_loss < best_loss:
        best_loss = avg_loss
        torch.save(model.state_dict(), "best_model.pth")
        print("Best model saved")

model.load_state_dict(torch.load("best_model.pth"))
model.eval()
predictions = []

with torch.no_grad():
    for images, image_ids in test_loader:
        images = images.to(DEVICE)
        outputs = model(images)
        preds = outputs.argmax(dim=1)

        for img_id, pred in zip(image_ids, preds.cpu().numpy()):
            pred_string = labeled_dataset.idx_to_label[pred]
            predictions.append([img_id.item() if torch.is_tensor(img_id) else img_id, pred_string])

submission = pd.DataFrame(predictions, columns=["ID", "prediction"])
submission.to_csv("submission.csv", index = False)
